# Goal: 
After adding a new feature to check for question openness(view post_quality_v1_1.ipynb), I concluded that there is only one row where there is question openness so I added 10 new data of similar type in different labels and let's see something changes or not

In [68]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import random

In [69]:
random.seed(42)
with open("../data/post_quality/post_data_v2.json","r",encoding="utf-8") as f:
    raw_data = json.load(f)


random.shuffle(raw_data)

df = pd.DataFrame(raw_data)
df.head()

,id,title,body,code,label,domain,source
0,47,Why are genetic differences in appearance and ...,"I mean, different human races have lived in di...",,1,priviledge,reddit
1,19,I changed a road sign to make my commute easie...,On my daily commute there was very inconvenien...,,0,personal-story,reddit
2,37,What’s a skill that sounds boring but is actua...,"Some skills don’t sound exciting at first, but...",,1,Thought-question,reddit
3,9,I can post!,So...\n😭💔😶‍🌫️🥱😮‍💨🙄😒😓😥☹️\nI'm tired.,,0,general-statement,reddit
4,8,My biggest fear is Alzheimer’s,"Not for me, for my mom.\nI don’t think I could...",,0,personal-story,reddit


In [70]:
df.shape

(75, 7)

# Data Cleaning

In [71]:
for col in df.columns:
    print(f"Number of null value in '{col}' column is {df[col].isnull().sum()}")

Number of null value in 'id' column is 0
Number of null value in 'title' column is 0
Number of null value in 'body' column is 0
Number of null value in 'code' column is 0
Number of null value in 'label' column is 0
Number of null value in 'domain' column is 0
Number of null value in 'source' column is 0


# Feature Engineering 1.0

In [72]:
df["text"] = df["title"] + "\n\n" + df['body']
df.head()

,id,title,body,code,label,domain,source,text
0,47,Why are genetic differences in appearance and ...,"I mean, different human races have lived in di...",,1,priviledge,reddit,Why are genetic differences in appearance and ...
1,19,I changed a road sign to make my commute easie...,On my daily commute there was very inconvenien...,,0,personal-story,reddit,I changed a road sign to make my commute easie...
2,37,What’s a skill that sounds boring but is actua...,"Some skills don’t sound exciting at first, but...",,1,Thought-question,reddit,What’s a skill that sounds boring but is actua...
3,9,I can post!,So...\n😭💔😶‍🌫️🥱😮‍💨🙄😒😓😥☹️\nI'm tired.,,0,general-statement,reddit,I can post!\n\nSo...\n😭💔😶‍🌫️🥱😮‍💨🙄😒😓😥☹️\nI'm ti...
4,8,My biggest fear is Alzheimer’s,"Not for me, for my mom.\nI don’t think I could...",,0,personal-story,reddit,"My biggest fear is Alzheimer’s\n\nNot for me, ..."


In [73]:
df['title_len'] = df['title'].str.len()
df['body_len'] = df['body'].str.len()
df['total_len'] = df['text'].str.len()

df[['title_len', 'body_len', 'total_len']].describe()

,title_len,body_len,total_len
count,75.000000,75.000000,75.00000
mean,77.320000,578.320000,657.64000
std,50.237856,804.042744,791.64952
min,11.000000,0.000000,36.00000
25%,46.500000,0.000000,131.00000
50%,66.000000,331.000000,377.00000
75%,94.000000,758.500000,851.50000
max,288.000000,5204.000000,5234.00000


In [74]:
df["body_to_title_ratio"] = df["body_len"] / (df["title_len"]+1)
df['num_para'] = df["body"].str.count("\n") + 1
df['num_questions'] = df["text"].str.count("\?")
df['num_exclamations'] = df['text'].str.count("!")
df['questions_per_100_words'] = df['num_questions'] / (len(df['text'].str.split().str.len())/100)


df["has_multiple_paras"] = (df['num_para'] > 1).astype(int)
df["has_questions"] = (df['num_questions'] > 0).astype(int)
df["has_exclamations"] = (df['num_exclamations'] > 0).astype(int)

In [75]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75 entries, 0 to 74
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       75 non-null     int64  
 1   title                    75 non-null     object 
 2   body                     75 non-null     object 
 3   code                     75 non-null     object 
 4   label                    75 non-null     int64  
 5   domain                   75 non-null     object 
 6   source                   75 non-null     object 
 7   text                     75 non-null     object 
 8   title_len                75 non-null     int64  
 9   body_len                 75 non-null     int64  
 10  total_len                75 non-null     int64  
 11  body_to_title_ratio      75 non-null     float64
 12  num_para                 75 non-null     int64  
 13  num_questions            75 non-null     int64  
 14  num_exclamations         75 

In [76]:
df = df.drop('total_len', axis=1)

df['log_num_para'] = np.log1p(df['num_para'])
df = df.drop('num_para',axis=1)

df["word_count"] = df['text'].str.split().str.len()
df['avg_word_len'] = df['text'].str.split().apply(
    lambda x: np.mean([len(w) for w in x]) if len(x) else 0
)

df['unique_word_ratio'] = df['text'].str.lower().str.split().apply(lambda x: len(set(x))) / df['word_count']
df['has_code'] = (df['code'].str.len() > 0).astype(int)

In [77]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75 entries, 0 to 74
Data columns (total 22 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       75 non-null     int64  
 1   title                    75 non-null     object 
 2   body                     75 non-null     object 
 3   code                     75 non-null     object 
 4   label                    75 non-null     int64  
 5   domain                   75 non-null     object 
 6   source                   75 non-null     object 
 7   text                     75 non-null     object 
 8   title_len                75 non-null     int64  
 9   body_len                 75 non-null     int64  
 10  body_to_title_ratio      75 non-null     float64
 11  num_questions            75 non-null     int64  
 12  num_exclamations         75 non-null     int64  
 13  questions_per_100_words  75 non-null     float64
 14  has_multiple_paras       75 

In [78]:
df.drop(['id','domain','source'],inplace=True, axis=1)

In [79]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75 entries, 0 to 74
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   title                    75 non-null     object 
 1   body                     75 non-null     object 
 2   code                     75 non-null     object 
 3   label                    75 non-null     int64  
 4   text                     75 non-null     object 
 5   title_len                75 non-null     int64  
 6   body_len                 75 non-null     int64  
 7   body_to_title_ratio      75 non-null     float64
 8   num_questions            75 non-null     int64  
 9   num_exclamations         75 non-null     int64  
 10  questions_per_100_words  75 non-null     float64
 11  has_multiple_paras       75 non-null     int64  
 12  has_questions            75 non-null     int64  
 13  has_exclamations         75 non-null     int64  
 14  log_num_para             75 

In [80]:
df = df.drop(columns=['body_to_title_ratio'])
df['log_btt'] = (
    np.log1p(df['body_len']) - np.log1p(df['title_len'])
)
df['log_btt_x_has_questions'] = (
    df['log_btt'] * df['has_questions']
)


In [81]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75 entries, 0 to 74
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   title                    75 non-null     object 
 1   body                     75 non-null     object 
 2   code                     75 non-null     object 
 3   label                    75 non-null     int64  
 4   text                     75 non-null     object 
 5   title_len                75 non-null     int64  
 6   body_len                 75 non-null     int64  
 7   num_questions            75 non-null     int64  
 8   num_exclamations         75 non-null     int64  
 9   questions_per_100_words  75 non-null     float64
 10  has_multiple_paras       75 non-null     int64  
 11  has_questions            75 non-null     int64  
 12  has_exclamations         75 non-null     int64  
 13  log_num_para             75 non-null     float64
 14  word_count               75 

In [82]:
df['question_density_per_para'] = (df['num_questions'] / (df['body'].str.count('\n') + 1))
df = df.drop(columns=['has_questions','num_questions'])

In [83]:
import re
FIRST_PERSON = {"i","me","my","mine","we","us","our","ours"}

def first_person_ratio(text):
    words = re.findall(r"\bw+\b",text.lower())
    if not words:
        return 0.0
    fp_count = sum(1 for w in words if w in FIRST_PERSON)
    return fp_count/ len(words)

df['first_person_ratio'] = df['text'].apply(first_person_ratio)

In [84]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75 entries, 0 to 74
Data columns (total 20 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   title                      75 non-null     object 
 1   body                       75 non-null     object 
 2   code                       75 non-null     object 
 3   label                      75 non-null     int64  
 4   text                       75 non-null     object 
 5   title_len                  75 non-null     int64  
 6   body_len                   75 non-null     int64  
 7   num_exclamations           75 non-null     int64  
 8   questions_per_100_words    75 non-null     float64
 9   has_multiple_paras         75 non-null     int64  
 10  has_exclamations           75 non-null     int64  
 11  log_num_para               75 non-null     float64
 12  word_count                 75 non-null     int64  
 13  avg_word_len               75 non-null     float64
 

# Feature Engineering 1.1: `is_open_ended_question: To measure openness`

In [85]:
OPEN_ENDED_PHRASES = [
    # Opinions & beliefs
    "what do you think",
    "what are your thoughts",
    "what is your perspective",
    "how would you describe",
    "in your opinion",
    "do you believe",
    "do you think that",
    "would you say that",

    # Feelings & reactions
    "how do you feel",
    "how does it make you feel",
    "what was your reaction",
    "what stood out to you",
    "how did it affect you",

    # Reasons & explanations
    "why do people",
    "why do we",
    "why do you think",
    "what do you think causes",
    "what leads people to",
    "what might explain",

    # Processes & behavior
    "how do people",
    "how do you usually",
    "how does this work",
    "how would someone",
    "what typically happens",

    # Possibilities & outcomes
    "what could happen",
    "what do you expect",
    "what comes after",
    "what might come next",
    "where do you see this going",

    # Reflection & evaluation
    "what does this mean to you",
    "how would you evaluate",
    "what matters most to you",
    "what do you find important",
    "what do you think is missing",

    # Comparisons & alternatives
    "how is this different from",
    "how is this similar to",
    "what other options are there",
    "what alternatives can you imagine",

    # Personal experience
    "have you ever noticed",
    "have you experienced",
    "what has your experience been",
    "can you tell me about",

    # Clarification & exploration
    "can you explain more",
    "can you elaborate",
    "what do you mean by",
    "how would you clarify"
]

In [86]:
def is_open_ended_question(title:str, body:str):
    text = (title + " " + body).lower()

    if "?" not in text:
        return 0
    
    for phrase in OPEN_ENDED_PHRASES:
        if phrase in text:
            return 1
    
    return 0

df['is_open_ended_question'] = df.apply(
    lambda row: is_open_ended_question(
        str(row['title']) if pd.notna(row['title']) else "",
        str(row['body']) if pd.notna(row['body']) else ""
    ),
    axis=1
)

In [87]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75 entries, 0 to 74
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   title                      75 non-null     object 
 1   body                       75 non-null     object 
 2   code                       75 non-null     object 
 3   label                      75 non-null     int64  
 4   text                       75 non-null     object 
 5   title_len                  75 non-null     int64  
 6   body_len                   75 non-null     int64  
 7   num_exclamations           75 non-null     int64  
 8   questions_per_100_words    75 non-null     float64
 9   has_multiple_paras         75 non-null     int64  
 10  has_exclamations           75 non-null     int64  
 11  log_num_para               75 non-null     float64
 12  word_count                 75 non-null     int64  
 13  avg_word_len               75 non-null     float64
 

In [88]:
df[df['is_open_ended_question'] == 1]

,title,body,code,label,text,title_len,body_len,num_exclamations,questions_per_100_words,has_multiple_paras,...,log_num_para,word_count,avg_word_len,unique_word_ratio,has_code,log_btt,log_btt_x_has_questions,question_density_per_para,first_person_ratio,is_open_ended_question
13,"Be honest, what do you think comes after death?",,,1,"Be honest, what do you think comes after death...",47,0,0,1.333333,0,...,0.693147,9,4.333333,1.000000,0,-3.871201,-3.871201,1.00,0.0,1
64,Why do people view body count as important in ...,"Just to be clear, I'll be using this definitio...",,2,Why do people view body count as important in ...,63,1488,0,1.333333,1,...,1.609438,256,5.066406,0.628906,0,3.146977,3.146977,0.25,0.0,1


### Observation:
Initial feature engineering failed because open-endedness was incorrectly operationalized as explicit conversational phrasing. A second-pass feature decomposed openness into structural and semantic signals, significantly improving recall for discussion-capable posts

In [89]:
FACT_SEEKING_STARTS = (
    # Identity / definition
    "who is", "who was", "what is", "what are", "what was",
    "define", "definition of", "meaning of",

    # Time
    "when did", "when was", "what year", "what date",

    # Location
    "where is", "where was", "where are",

    # Quantity / measurement
    "how many", "how much", "how long", "how far", "how old",

    # Lists / enumeration
    "what are the types of", "what are the kinds of",
    "list the", "name the",

    # Comparatively factual
    "which is", "which are",

    # Capital / super common trivia patterns
    "what is the capital", "what is the population",
)
OPEN_QUESTION_STARTS = (
    # Reasoning & explanation
    "why", "how", "how does", "how do", "how should",

    # Exploration & interpretation
    "what do you think", "what do you believe",
    "what does it mean", "what are the implications of",

    # Hypotheticals
    "what would happen if", "what if",
    "would", "could", "should",

    # Advice / judgment
    "do you think", "do you believe", "do you agree",
    "is it better to", "is it good to",

    # Evaluation & comparison
    "which is better", "which is worse",
    "what is the best way to", "what is the worst way to",

    # Process / creativity
    "how can", "how might", "how would you",
)


In [90]:
def is_open_ended_question_v2(title: str, body: str):
    text = (title + " " + body).lower().strip()

    if "?" not in text:
        return 0

    # Remove obvious fact-seeking
    for fs in FACT_SEEKING_STARTS:
        if text.startswith(fs):
            return 0

    # Open-ended structural cues
    for oq in OPEN_QUESTION_STARTS:
        if text.startswith(oq):
            return 1

    return 0


In [91]:

df['is_open_ended_question'] = df.apply(
    lambda row: is_open_ended_question_v2(
        str(row['title']) if pd.notna(row['title']) else "",
        str(row['body']) if pd.notna(row['body']) else ""
    ),
    axis=1
)

In [92]:
df['is_open_ended_question'].value_counts()


is_open_ended_question
0    60
1    15
Name: count, dtype: int64

In [93]:
## Let's build a helper function to make error_df
import pandas as pd

def build_error_df(df_test, y_true, y_pred, model_name):
    out = df_test.copy()
    out['true_label'] = y_true
    out['pred_label'] = y_pred
    out['correct'] = y_true == y_pred
    out['model'] = model_name
    return out

In [94]:
## Generic function to split df in same test and train
from sklearn.model_selection import train_test_split


def split_df(
    df,
    label_col,
    test_size=0.2,
    random_state=52,
    stratify=True
):
    """
    Splits dataframe into train and test sets.

    Returns
    -------
    df_train : pd.DataFrame
    df_test : pd.DataFrame
    """

    df = df.reset_index(drop=True)

    stratify_col = df[label_col] if stratify else None

    train_idx, test_idx = train_test_split(
        df.index,
        test_size=test_size,
        random_state=random_state,
        stratify=stratify_col
    )

    df_train = df.loc[train_idx]
    df_test = df.loc[test_idx]

    return df_train, df_test


In [95]:
## Running numerical only model
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,accuracy_score


def run_numerical_model(
    df_train,
    df_test,
    label_col,
    num_cols=None,
    model=None,
    scaler=None,
    model_name="numerical_only"
):
    """
    Trains and evaluates a numerical-only classification model.

    Returns
    -------
    dict with:
        - model
        - scaler
        - classification_report
        - errors_df
        - y_pred
    """

    if num_cols is None:
        num_cols = df_train.select_dtypes(include="number").columns.tolist()
        num_cols.remove(label_col)

    if scaler is None:
        scaler = StandardScaler()

    if model is None:
        model = LogisticRegression(
            max_iter=2000,
            class_weight="balanced"
        )

    X_train = scaler.fit_transform(df_train[num_cols])
    X_test = scaler.transform(df_test[num_cols])

    y_train = df_train[label_col]
    y_test = df_test[label_col]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    report = classification_report(y_test, y_pred)

    errors_df = build_error_df(
        df_test=df_test,
        y_true=y_test,
        y_pred=y_pred,
        model_name=model_name
    )

    errors_df['error_type'] = ''
    errors_df['error_subtype'] = ''
    errors_df['notes'] = ''

    return {
        "model": model,
        "scaler": scaler,
        "classification_report": report,
        "errors_df": errors_df,
        "accuracy":accuracy_score(y_test,y_pred),
        "y_pred": y_pred
    }


In [96]:
df_train_generic, df_test_generic = split_df(df,'label')
result_num_v2 = run_numerical_model(df_train_generic,df_test_generic,'label')
df_train_generic.shape,df_test_generic.shape

((60, 21), (15, 21))

In [97]:
wrong_num_v2 = result_num_v2['errors_df']
wrong_num_v2[(wrong_num_v2['correct'] == False) & (wrong_num_v2['is_open_ended_question'] == 1)].shape


(2, 28)

In [98]:
## Total open ended questions in test set
df_test_generic[df_test_generic['is_open_ended_question'] == 1].shape

(5, 21)

### Observation:
#### With using `is_open_ended_question` I it got 2 rows wrongly predicted open ended question in test cases out of 5 test cases. Around 60% accuracy

### Suppose dropping the `isOpenQuestion` param and test how much is classified?

In [99]:
df_train_generic_v2 = df_train_generic.drop('is_open_ended_question',axis=1)
df_test_generic_v2 = df_test_generic.drop('is_open_ended_question',axis=1)

In [100]:
result_num_v3 = run_numerical_model(df_train_generic_v2,df_test_generic_v2,'label')
df_train_generic_v2.shape,df_test_generic_v2.shape

((60, 20), (15, 20))

In [101]:
## Checking how many wrongly predicted open ended questions are there in df

wrong_num_v3 = result_num_v3['errors_df']
wrong_num_v3[
    (wrong_num_v3['correct'] == False) &
    (df.loc[wrong_num_v3.index, 'is_open_ended_question'] == 1)
].shape[0]


1

### Observation:
#### Without using `is_open_ended_question` I it got 1 rows wrongly predicted open ended question in test cases out of 5 test cases. Around 80% accuracy

## 5-Fold Cross Validation
### Goal:
I need to answer this important question before moving to error taxonomy: **Does `is_open_ended_question` help on average, not just on one lucky split?**

In [102]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

## List to store results
acc_with_feature = []
acc_without_feature = []

open_acc_with = []
open_acc_without = []

for fold, (train_idx, test_idx) in enumerate(skf.split(df, df['label'])):
    train_df = df.iloc[train_idx].copy()
    test_df = df.iloc[test_idx].copy()

    ## Training with the is_open_ended_question variable
    result_with = run_numerical_model(train_df,test_df,'label')
    acc_with_feature.append(result_with['accuracy'])

    errors_with = result_with['errors_df']

    open_test = test_df[test_df['is_open_ended_question']==1]

    if len(open_test)>0:
        wrong_open = errors_with[
        (errors_with['correct'] == False) &
        (errors_with.index.isin(open_test.index))].shape[0]

        open_acc_with.append(
            1 - wrong_open / len(open_test)
        )

    ## Now without open-ended question feature
    train_df_no = train_df.drop(
        columns=['is_open_ended_question'],
        errors='ignore'
    )
    test_df_no = test_df.drop(
        columns=['is_open_ended_question'],
        errors='ignore'
    )

    result_without = run_numerical_model(
        train_df_no,test_df_no,'label'
    )

    acc_without_feature.append(result_without['accuracy'])

    errors_without = result_without['errors_df']

    if len(open_test) > 0:
        wrong_open = errors_without[
            (errors_without['correct'] == False) &
            (errors_without.index.isin(open_test.index))
        ].shape[0]

        open_acc_without.append(
            1 - wrong_open / len(open_test)
        )

print("Overall accuracy WITH feature:",
      np.mean(acc_with_feature))

print("Overall accuracy WITHOUT feature:",
      np.mean(acc_without_feature))

print("Open-ended accuracy WITH feature:",
      np.mean(open_acc_with))

print("Open-ended accuracy WITHOUT feature:",
      np.mean(open_acc_without))




Overall accuracy WITH feature: 0.48
Overall accuracy WITHOUT feature: 0.4666666666666667
Open-ended accuracy WITH feature: 0.4833333333333334
Open-ended accuracy WITHOUT feature: 0.6166666666666668


## Observation:
“Binary `is_open_ended_question` feature hurts open-ended subset accuracy under 5-fold CV → dropped.”

In [103]:
df_base = df.drop(columns=['is_open_ended_question'], errors='ignore')

### Doing error taxonomy on open ended questions across 5 fold validation

In [104]:
from sklearn.model_selection import StratifiedKFold
import pandas as pd

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

all_errors = []

for fold, (train_idx, test_idx) in enumerate(skf.split(df_base,df_base['label'])):
    train_df = df_base.iloc[train_idx].copy()
    test_df = df_base.iloc[test_idx].copy()

    result = run_numerical_model(train_df,test_df,'label')

    errors_df = result['errors_df'].copy()
    errors_df['fold'] = fold ## Let's do the tracking of fold

    all_errors.append(errors_df)

    errors_all = pd.concat(all_errors)

In [105]:
errors_all

,title,body,code,label,text,title_len,body_len,num_exclamations,questions_per_100_words,has_multiple_paras,...,question_density_per_para,first_person_ratio,true_label,pred_label,correct,model,error_type,error_subtype,notes,fold
13,"Be honest, what do you think comes after death?",,,1,"Be honest, what do you think comes after death...",47,0,0,1.333333,0,...,1.000000,0.0,1,0,False,numerical_only,,,,0
28,What poor people food was ruined because of ri...,,,0,What poor people food was ruined because of ri...,56,0,0,1.333333,0,...,1.000000,0.0,0,0,True,numerical_only,,,,0
29,marketo form validation with fetch,I added a fetch function to do some hidden val...,form.onSubmit(async function(form) {\n let ...,2,marketo form validation with fetch\n\nI added ...,34,248,0,0.000000,0,...,0.000000,0.0,2,2,True,numerical_only,,,,0
31,A simple rule to help build your own thing,Let me start off by saying that work as a web ...,,0,A simple rule to help build your own thing\n\n...,42,1243,0,1.333333,1,...,0.076923,0.0,0,1,False,numerical_only,,,,0
32,Learn programming as an absolute beginner,I’m a complete outsider here I’ve always wante...,,1,Learn programming as an absolute beginner\n\nI...,41,299,0,4.000000,0,...,3.000000,0.0,1,2,False,numerical_only,,,,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54,How do I (21M) ask my GF (20F) to take a showe...,This is my first relationship. I (21M) have be...,,1,How do I (21M) ask my GF (20F) to take a showe...,76,755,0,2.666667,1,...,0.500000,0.0,1,2,False,numerical_only,,,,4
58,What is best crime thriller series to watch on...,,,0,What is best crime thriller series to watch on...,55,0,0,1.333333,0,...,1.000000,0.0,0,0,True,numerical_only,,,,4
63,How do some men get so many women when they ar...,"I feel like we've all seen it. Like, the guy w...",,2,How do some men get so many women when they ar...,86,415,0,4.000000,1,...,1.500000,0.0,2,2,True,numerical_only,,,,4
65,Men's public restrooms are laid out all wrong....,,,0,Men's public restrooms are laid out all wrong....,145,0,0,0.000000,0,...,0.000000,0.0,0,1,False,numerical_only,,,,4


In [106]:
errors_all['is_open_ended'] = df.loc[errors_all.index, 'is_open_ended_question']
errors_all

,title,body,code,label,text,title_len,body_len,num_exclamations,questions_per_100_words,has_multiple_paras,...,first_person_ratio,true_label,pred_label,correct,model,error_type,error_subtype,notes,fold,is_open_ended
13,"Be honest, what do you think comes after death?",,,1,"Be honest, what do you think comes after death...",47,0,0,1.333333,0,...,0.0,1,0,False,numerical_only,,,,0,0
28,What poor people food was ruined because of ri...,,,0,What poor people food was ruined because of ri...,56,0,0,1.333333,0,...,0.0,0,0,True,numerical_only,,,,0,0
29,marketo form validation with fetch,I added a fetch function to do some hidden val...,form.onSubmit(async function(form) {\n let ...,2,marketo form validation with fetch\n\nI added ...,34,248,0,0.000000,0,...,0.0,2,2,True,numerical_only,,,,0,0
31,A simple rule to help build your own thing,Let me start off by saying that work as a web ...,,0,A simple rule to help build your own thing\n\n...,42,1243,0,1.333333,1,...,0.0,0,1,False,numerical_only,,,,0,0
32,Learn programming as an absolute beginner,I’m a complete outsider here I’ve always wante...,,1,Learn programming as an absolute beginner\n\nI...,41,299,0,4.000000,0,...,0.0,1,2,False,numerical_only,,,,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54,How do I (21M) ask my GF (20F) to take a showe...,This is my first relationship. I (21M) have be...,,1,How do I (21M) ask my GF (20F) to take a showe...,76,755,0,2.666667,1,...,0.0,1,2,False,numerical_only,,,,4,1
58,What is best crime thriller series to watch on...,,,0,What is best crime thriller series to watch on...,55,0,0,1.333333,0,...,0.0,0,0,True,numerical_only,,,,4,0
63,How do some men get so many women when they ar...,"I feel like we've all seen it. Like, the guy w...",,2,How do some men get so many women when they ar...,86,415,0,4.000000,1,...,0.0,2,2,True,numerical_only,,,,4,1
65,Men's public restrooms are laid out all wrong....,,,0,Men's public restrooms are laid out all wrong....,145,0,0,0.000000,0,...,0.0,0,1,False,numerical_only,,,,4,0


In [107]:
open_errors = errors_all[(errors_all['is_open_ended'] == 1)&(errors_all['correct'] == False)]
open_errors

,title,body,code,label,text,title_len,body_len,num_exclamations,questions_per_100_words,has_multiple_paras,...,first_person_ratio,true_label,pred_label,correct,model,error_type,error_subtype,notes,fold,is_open_ended
34,How would you feel if your country banned Burk...,,,1,How would you feel if your country banned Burk...,73,0,0,1.333333,0,...,0.0,1,0,False,numerical_only,,,,0,1
8,"How can i stop idealising a better, fulfilling...",Title might be confusing but I just want to kn...,,1,"How can i stop idealising a better, fulfilling...",79,1090,0,2.666667,1,...,0.0,1,2,False,numerical_only,,,,1,1
57,"how do you grab life by its horn and live a ""s...",,,2,"how do you grab life by its horn and live a ""s...",85,0,0,1.333333,0,...,0.0,2,0,False,numerical_only,,,,2,1
54,How do I (21M) ask my GF (20F) to take a showe...,This is my first relationship. I (21M) have be...,,1,How do I (21M) ask my GF (20F) to take a showe...,76,755,0,2.666667,1,...,0.0,1,2,False,numerical_only,,,,4,1


In [108]:
open_errors[['title', 'body', 'true_label', 'pred_label', 'fold']]


,title,body,true_label,pred_label,fold
34,How would you feel if your country banned Burk...,,1,0,0
8,"How can i stop idealising a better, fulfilling...",Title might be confusing but I just want to kn...,1,2,1
57,"how do you grab life by its horn and live a ""s...",,2,0,2
54,How do I (21M) ask my GF (20F) to take a showe...,This is my first relationship. I (21M) have be...,1,2,4


In [109]:
open_errors['error_reason'] = ""


C:\Users\prakh\AppData\Local\Temp\ipykernel_24920\739410073.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  open_errors['error_reason'] = ""


In [110]:
open_errors[['title', 'body', 'true_label', 'pred_label', 'fold']]


,title,body,true_label,pred_label,fold
34,How would you feel if your country banned Burk...,,1,0,0
8,"How can i stop idealising a better, fulfilling...",Title might be confusing but I just want to kn...,1,2,1
57,"how do you grab life by its horn and live a ""s...",,2,0,2
54,How do I (21M) ask my GF (20F) to take a showe...,This is my first relationship. I (21M) have be...,1,2,4


In [111]:
errors_all[errors_all['is_open_ended'] == 1].shape[0]


15

## Observation – Open-ended question miscalibration
Across all misclassified open-ended questions, the model shows a strong structural bias based on post length rather than intent or engagement potential.

- **Title-only open-ended posts are consistently undervalued and predicted as the lowest quality class (label 0), regardless of their true label.**
- **Open-ended posts containing both title and body text are consistently overvalued and predicted as the highest quality class (label 2), even when the true label is moderate (label 1).**

This indicates that the model relies heavily on surface-level text length cues and fails to correctly interpret open-ended intent as an independent signal of post quality.

### Error Category Decision
This error subtype is marked as **Category D**.

#### Reasoning:
* The error stems from a fundamental limitation of the current feature space and model capacity rather than a simple missing heuristic.
* Resolving this issue would likely require:
    - richer semantic understanding of question intent,
    - discourse-level features, or
    - substantially more labeled data focused on open-ended question structure.
* Given the small number of remaining errors and the limited expected performance gain, further optimization is not cost-effective at this stage.

#### Action:
Accepted the limitation, documented it clearly, and moving forward to analyze other error subtypes.

In [112]:
df_base.to_csv("../data/post_quality/processed/post_quality_fe_v2.csv",index=False)

In [113]:
df_new = pd.read_csv("../data/post_quality/processed/post_quality_fe_v2.csv")

In [114]:
df_base['first_person_ratio'].unique()

array([0.])

In [115]:
df_base.columns

Index(['title', 'body', 'code', 'label', 'text', 'title_len', 'body_len',
       'num_exclamations', 'questions_per_100_words', 'has_multiple_paras',
       'has_exclamations', 'log_num_para', 'word_count', 'avg_word_len',
       'unique_word_ratio', 'has_code', 'log_btt', 'log_btt_x_has_questions',
       'question_density_per_para', 'first_person_ratio'],
      dtype='object')

**Note**: `first_person_ratio` was previously inactive due to a regex bug and has been corrected. Since this feature does not directly address the open-ended miscalibration error (Category D), prior conclusions remain unchanged.

In [116]:
import re

FIRST_PERSON = {"i", "me", "my", "mine", "we", "us", "our", "ours"}

def first_person_ratio(text):
    if not isinstance(text, str):
        return 0.0

    words = re.findall(r"\b\w+\b", text.lower())
    if not words:
        return 0.0

    fp_count = sum(1 for w in words if w in FIRST_PERSON)
    return fp_count / len(words)

df['first_person_ratio'] = df['text'].apply(first_person_ratio)


In [117]:
df['first_person_ratio'].describe()


count    75.000000
mean      0.047696
std       0.052373
min       0.000000
25%       0.000000
50%       0.039604
75%       0.069714
max       0.285714
Name: first_person_ratio, dtype: float64

In [118]:
df[df['first_person_ratio'] > 0][['text', 'first_person_ratio']].head()


,text,first_person_ratio
0,Why are genetic differences in appearance and ...,0.015625
1,I changed a road sign to make my commute easie...,0.065385
2,What’s a skill that sounds boring but is actua...,0.014925
3,I can post!\n\nSo...\n😭💔😶‍🌫️🥱😮‍💨🙄😒😓😥☹️\nI'm ti...,0.285714
4,"My biggest fear is Alzheimer’s\n\nNot for me, ...",0.142857


In [119]:
df.drop('is_open_ended_question',inplace=True,axis=1)

In [120]:
df.to_csv("../data/post_quality/processed/post_quality_fe2_fixed.csv", index=False)
